# Faruq-v3 ACMC2 — paired three-seed confirmation

Konfirmasi validation-only ACMC2 entropy+margin. Seed 42 direuse dari screening yang sudah PASS. Seed 123 dan 2026 hanya melatih ACMC2 dari checkpoint D0 yang sama dengan paired ACMC1 sebelumnya; D0, D0FT, dan ACMC1 tidak dilatih ulang. Test tidak tersedia atau dibuka.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/acmc2-margin-gate'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO) + '[dev]'], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('REPO:', REPO)
print('BRANCH:', BRANCH)

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan GPU Colab.'
required = (
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/val_reports/acmc1_paired_optimization_confirmation.json',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed123/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed2026/weights/best.pt',
    'experiments/faruq-v3-acmc2-entropy-margin-v1/val_reports/acmc2_seed42_screening.json',
)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=required)
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
ACMC1_PAIRED_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-acmc-paired-confirmation-v1'
ACMC1_PAIRED_SUMMARY = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-paired-confirmation-v1/val_reports/acmc1_paired_optimization_confirmation.json')
ACMC2_SEED42_SUMMARY = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc2-entropy-margin-v1/val_reports/acmc2_seed42_screening.json')
for seed in (123, 2026):
    require_project_artifact(PROJECT_ROOT, f'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed{seed}/weights/best.pt')

DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-acmc2-paired-confirmation-v1'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'

for seed in (123, 2026):
    last = OUTPUT_ROOT / f'ACMC2_seed{seed}/weights/last.pt'
    best = OUTPUT_ROOT / f'ACMC2_seed{seed}/weights/best.pt'
    status = 'COMPLETE' if best.is_file() else ('RESUME' if last.is_file() else 'START')
    print(f'ACMC2 seed {seed}: {status}')
print('GPU    :', torch.cuda.get_device_name(0))
print('PROJECT:', PROJECT_ROOT)
print('OUTPUT :', OUTPUT_ROOT)

In [ ]:
tests = [
    'tests/test_acmc2_entropy_margin.py',
    'tests/test_acmc2_screening.py',
    'tests/test_acmc2_paired_confirmation.py',
    'tests/test_ambiguity_multilevel.py',
]
print('PRE-TRAINING TESTS:', ' '.join(tests), flush=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', *tests], cwd=REPO, check=True)
print('PRE-TRAINING TESTS: PASS')

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_acmc2_paired_confirmation',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--acmc1-paired-summary', str(ACMC1_PAIRED_SUMMARY),
    '--acmc1-paired-root', str(ACMC1_PAIRED_ROOT),
    '--acmc2-seed42-summary', str(ACMC2_SEED42_SUMMARY),
    '--output-root', str(OUTPUT_ROOT),
    '--seeds', '123', '2026', '--device', '0', '--authorize-training',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'Konfirmasi ACMC2 gagal, return code={return_code}; traceback lengkap tercetak di atas.')

In [ ]:
import json, pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'val_reports/acmc2_paired_optimization_confirmation.json'
assert SUMMARY.is_file(), f'Konfirmasi belum selesai: {SUMMARY}'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
assert result['test_opened'] is False
rows = [{'metric': metric, **values} for metric, values in result['aggregate'].items()]
formatters = {name: '{:.2%}' for name in (
    'd0_mean', 'd0ft_mean', 'acmc1_mean', 'acmc2_mean',
    'acmc2_vs_d0ft_mean', 'acmc2_vs_d0ft_min',
    'acmc2_vs_acmc1_mean', 'acmc2_vs_acmc1_min',
)}
display(pd.DataFrame(rows).style.format(formatters))
for seed in (42, 123, 2026):
    row = {'seed': seed, **result['per_seed'][str(seed)]['results']}
    print(f'SEED {seed}:', row)
print('CRITERIA:', result['criteria'])
print('DECISION:', result['decision'])
print('NEXT    :', result['next_action'])
print('SUMMARY :', SUMMARY)
print('Kirim tabel aggregate, per-seed, criteria, dan decision. Jangan membuka test secara manual.')